In [1]:
# ============================================================
# BattingEdge V9.5 - CNN-LSTM HYBRID Model Training
# Features: 99 raw pose + 8 angles (NO velocities)
# Target: 90%+ Accuracy (Combines Spatial + Temporal)
# ============================================================

# ✅ FIX: Import Pandas first
import pandas as pd 
import numpy as np
import pickle
from pathlib import Path
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import json
import joblib

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Conv1D, MaxPooling1D, LSTM
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam

# ================= CONFIG =================
FEATURE_DIR = Path(r"D:\Users\Anoshia\BattingEdge_FYP\features")
MODEL_DIR   = Path(r"D:\Users\Anoshia\BattingEdge_FYP\backend\models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

ALGO_NAME = "cnn_lstm"

EPOCHS = 60
BATCH_SIZE = 16
LEARNING_RATE = 1e-4  # Hybrid models need careful tuning
# ==========================================

print("="*70)
print(f"BATTINGEDGE V9.5 - MODEL TRAINING ({ALGO_NAME.upper()})")
print("="*70)
print()

# ================= LOAD DATA =================
print("📂 Loading data...")

try:
    X_train = np.load(FEATURE_DIR / "X_train.npy")
    y_train = np.load(FEATURE_DIR / "y_train.npy")
    X_val   = np.load(FEATURE_DIR / "X_val.npy")
    y_val   = np.load(FEATURE_DIR / "y_val.npy")
    X_test  = np.load(FEATURE_DIR / "X_test.npy")
    y_test  = np.load(FEATURE_DIR / "y_test.npy")

    with open(FEATURE_DIR / "classes.pkl", "rb") as f:
        CLASSES = pickle.load(f)
except FileNotFoundError as e:
    print(f"\n❌ CRITICAL ERROR: Could not find data files.")
    print(f"   Checked directory: {FEATURE_DIR}")
    raise e

num_classes = len(CLASSES)
T, F = X_train.shape[1], X_train.shape[2]

print(f"Train: {X_train.shape[0]} samples")
print(f"Val:   {X_val.shape[0]} samples")
print(f"Test:  {X_test.shape[0]} samples")
print(f"Classes: {CLASSES}")
print(f"Features per frame: {F}")
print()

# ================= CRITICAL: SCALING =================
print("⚖️  Applying StandardScaler...")

scaler = StandardScaler()

# Fit on training data (flatten to 2D)
N_train = X_train.shape[0]
X_train_2d = X_train.reshape(N_train * T, F)
scaler.fit(X_train_2d)

def scale_data(X):
    N, T, F = X.shape
    X_2d = X.reshape(N * T, F)
    X_scaled = scaler.transform(X_2d)
    return X_scaled.reshape(N, T, F)

X_train = scale_data(X_train)
X_val   = scale_data(X_val)
X_test  = scale_data(X_test)

# Save scaler
scaler_path = MODEL_DIR / f"scaler_V9_5_{ALGO_NAME}.pkl"
classes_path = MODEL_DIR / f"classes_V9_5_{ALGO_NAME}.pkl"
joblib.dump(scaler, scaler_path)
joblib.dump(CLASSES, classes_path)
print(f"✅ Scaler saved: {scaler_path.name}")
print()

# ================= CLASS WEIGHTS =================
print("⚖️  Computing class weights...")

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(num_classes),
    y=y_train
)
class_weight_dict = dict(enumerate(class_weights))
print("   Weights loaded.")
print()

# ================= MODEL: CNN-LSTM HYBRID =================
print("🏗️  Building CNN-LSTM model...")

model = Sequential([
    # CNN Block: Extracts local features from the sequence
    # "filters=64" looks for 64 different patterns (like a sudden elbow movement)
    Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=(T, F)),
    BatchNormalization(),
    MaxPooling1D(pool_size=2), # Reduces sequence length (Downsampling)
    Dropout(0.3),

    # CNN Block 2
    Conv1D(filters=128, kernel_size=3, activation='relu'),
    BatchNormalization(),
    MaxPooling1D(pool_size=2),
    Dropout(0.3),

    # LSTM Block: Learns the temporal sequence of the extracted features
    LSTM(128, return_sequences=False), # False because we want one prediction per video
    Dropout(0.4),

    # Classification Head
    Dense(64, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()
print()

# ================= CALLBACKS =================
print("⚙️  Setting up callbacks...")

checkpoint_path = MODEL_DIR / f"battingedge_V9_5_{ALGO_NAME}_best.keras"

callbacks_list = [
    EarlyStopping(monitor="val_loss", patience=12, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=6, min_lr=1e-6, verbose=1),
    ModelCheckpoint(str(checkpoint_path), monitor="val_accuracy", save_best_only=True, verbose=1)
]

print(f"✅ Checkpoint: {checkpoint_path.name}")
print()

# ================= TRAINING =================
print("="*70)
print("🚀 STARTING TRAINING")
print("="*70)
print()

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weight_dict,
    callbacks=callbacks_list,
    verbose=1
)

print()
print("="*70)
print("✅ TRAINING COMPLETE")
print("="*70)
print()

# Save final model
final_path = MODEL_DIR / f"battingedge_V9_5_{ALGO_NAME}_final.keras"
model.save(final_path)
print(f"💾 Saved final model: {final_path.name}")
print()

# ================= EVALUATION =================
print("="*70)
print("📊 EVALUATING ON TEST SET")
print("="*70)
print()

# Load best model
best_model = tf.keras.models.load_model(str(checkpoint_path))
print(f"✅ Loaded best model from: {checkpoint_path.name}")
print()

# Predict
y_pred = np.argmax(best_model.predict(X_test, verbose=0), axis=1)

# Classification report
print("📋 CLASSIFICATION REPORT:")
print()
report = classification_report(y_test, y_pred, target_names=CLASSES, digits=3)
print(report)

report_path = MODEL_DIR / f"report_V9_5_{ALGO_NAME}.txt"
with open(report_path, "w") as f:
    f.write(report)
print(f"✅ Report saved: {report_path.name}")
print()

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)

print("🔢 CONFUSION MATRIX:")
print("   (Rows = True, Cols = Predicted)")
print()
print("        ", "  ".join([f"{cls[:4]:>4s}" for cls in CLASSES]))
for i, cls in enumerate(CLASSES):
    print(f"{cls[:8]:8s}", "  ".join([f"{cm[i,j]:4d}" for j in range(num_classes)]))
print()

# Per-class accuracy
print("📈 PER-CLASS ACCURACY:")
print()
for i, cls in enumerate(CLASSES):
    correct = cm[i, i]
    total = cm[i, :].sum()
    accuracy = (correct / total * 100) if total > 0 else 0
    print(f"   {cls:15s}: {correct:3d}/{total:3d} = {accuracy:5.1f}%")

overall_acc = np.trace(cm) / np.sum(cm) * 100
print()
print(f"   {'OVERALL':15s}: {np.trace(cm):3d}/{np.sum(cm):3d} = {overall_acc:5.2f}%")
print()

# Major confusions
print("🔍 MAJOR CONFUSIONS (>3 cases):")
print()
confusions = []
for i in range(num_classes):
    for j in range(num_classes):
        if i != j and cm[i, j] > 3:
            confusions.append((CLASSES[i], CLASSES[j], cm[i, j]))

if confusions:
    confusions.sort(key=lambda x: x[2], reverse=True)
    for true_cls, pred_cls, count in confusions:
        print(f"   {true_cls:15s} → {pred_cls:15s}: {count} cases")
else:
    print("   ✅ No major confusions!")
print()

# ================= VISUALIZATIONS =================
print("📊 Generating visualizations...")

# Confusion matrix heatmap
cm_path = MODEL_DIR / f"confusion_matrix_V9_5_{ALGO_NAME}.png"
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASSES, yticklabels=CLASSES)
plt.title(f"Confusion Matrix - V9.5 ({ALGO_NAME})", fontsize=14, fontweight='bold')
plt.ylabel("True Label")
plt.xlabel("Predicted Label")
plt.tight_layout()
plt.savefig(cm_path, dpi=300)
print(f"   ✅ Saved: {cm_path.name}")
plt.close()

# Training history
history_path = MODEL_DIR / f"training_history_V9_5_{ALGO_NAME}.png"
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train')
plt.plot(history.history['val_accuracy'], label='Val')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Model Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train')
plt.plot(history.history['val_loss'], label='Val')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Model Loss')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(history_path, dpi=300)
print(f"   ✅ Saved: {history_path.name}")
plt.close()

print()

# ================= METADATA =================
print("="*70)
print(f"🎉 V9.5 ({ALGO_NAME}) TRAINING COMPLETE")
print("="*70)
print()

# Save metadata
metadata = {
    "version": "V9.5",
    "algorithm": ALGO_NAME,
    "features": "99 raw pose + 8 angles (no velocities)",
    "classes": CLASSES,
    "test_accuracy": float(overall_acc),
    "per_class_accuracy": {
        CLASSES[i]: float((cm[i,i] / cm[i,:].sum() * 100) if cm[i,:].sum() > 0 else 0)
        for i in range(num_classes)
    },
    "hyperparameters": {
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE
    }
}

metadata_path = MODEL_DIR / f"metadata_V9_5_{ALGO_NAME}.json"
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)

print("📦 Saved artifacts:")
print(f"   1. Best model: {checkpoint_path.name}")
print(f"   2. Final model: {final_path.name}")
print(f"   3. Metadata: {metadata_path.name}")
print()

if overall_acc >= 85:
    print("✨ EXCELLENT RESULT! The Hybrid architecture works.")
elif overall_acc >= 81:
    print("✅ GOOD RESULT. Competitive with standard LSTM.")
else:
    print("⚠️ Below target. Check CNN kernel size.")

print("="*70)

BATTINGEDGE V9.5 - MODEL TRAINING (CNN_LSTM)

📂 Loading data...
Train: 3007 samples
Val:   388 samples
Test:  378 samples
Classes: ['Cover Drive', 'Cut Shot', 'Defense', 'Pull Shot', 'Sweep Shot']
Features per frame: 107

⚖️  Applying StandardScaler...
✅ Scaler saved: scaler_V9_5_cnn_lstm.pkl

⚖️  Computing class weights...
   Weights loaded.

🏗️  Building CNN-LSTM model...


d:\Users\Anoshia\BattingEdge_FYP\venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 48, 64)         │        20,608 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 48, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 24, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 24, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 22, 128)        │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 22, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 11, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 11, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 128)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5)              │           325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 186,501 (728.52 KB)

 Trainable params: 185,989 (726.52 KB)

 Non-trainable params: 512 (2.00 KB)


⚙️  Setting up callbacks...
✅ Checkpoint: battingedge_V9_5_cnn_lstm_best.keras

🚀 STARTING TRAINING

Epoch 1/60
187/188 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.2311 - loss: 2.2474
Epoch 1: val_accuracy improved from None to 0.43299, saving model to D:\Users\Anoshia\BattingEdge_FYP\backend\models\battingedge_V9_5_cnn_lstm_best.keras
188/188 ━━━━━━━━━━━━━━━━━━━━ 34s 51ms/step - accuracy: 0.2754 - loss: 2.0837 - val_accuracy: 0.4330 - val_loss: 1.4133 - learning_rate: 1.0000e-04
Epoch 2/60
188/188 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.3522 - loss: 1.8179
Epoch 2: val_accuracy improved from 0.43299 to 0.52320, saving model to D:\Users\Anoshia\BattingEdge_FYP\backend\models\battingedge_V9_5_cnn_lstm_best.keras
188/188 ━━━━━━━━━━━━━━━━━━━━ 7s 38ms/step - accuracy: 0.3701 - loss: 1.7514 - val_accuracy: 0.5232 - val_loss: 1.2200 - learning_rate: 1.0000e-04
Epoch 3/60
188/188 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.4094 - loss: 1.6353
Epoch 3: val_accuracy improved fr